In [5]:
# 全局底层库线程限制，避免并行抢占
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# 基础数据处理库
import re
import numpy as np
import pandas as pd
import torch

# 机器学习工具库
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# 全局随机种子固定，保证结果可复现
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [6]:
# ===================== 列名常量 1:1匹配简易ASR模型表 =====================
COL_COATING = "涂层X"
COL_SYNTHESIS = "合成方法"
COL_PREPARATION = "制备方法"
COL_PHASE_TRANS_RED = "相转温度(还原气氛)"
COL_PHASE_TRANS_AIR = "相转温度（空气）（℃）"
COL_SUBSTRATE = "连接体基体X"
COL_THICKNESS = "涂层厚度(μm)X"
COL_COND_TEMP = "电导率测试温度"
COL_CONDUCTIVITY = "电导率"
COL_ASR_TEMP = "ASR测试温度(℃)X"
COL_ASR_TIME = "ASR测试时间(h)"
COL_ASR = "ASR(mΩ cm²)Y"
COL_ASR_INIT = "起始ASR"
COL_ACTIVATION_E = "电导率活化能(eV)"
COL_DOI = "DOI"

CATEGORICAL_COLS = [COL_SYNTHESIS, COL_PREPARATION, COL_SUBSTRATE]
NUMERIC_COLS = [
    COL_PHASE_TRANS_RED, COL_PHASE_TRANS_AIR, COL_THICKNESS,
    COL_ASR_TEMP, COL_ASR_TIME, COL_ASR_INIT, COL_ACTIVATION_E
]
LABEL_COL = COL_ASR
DROP_COLS = [COL_DOI, COL_COND_TEMP, COL_CONDUCTIVITY]

# ===================== 随机样本划分（允许同一涂层分布在训练/测试集） =====================
def split_by_formula(df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    # 按样本随机抽样划分，不再按涂层配方强制隔离
    test_df = df.sample(frac=test_size, random_state=random_state)
    train_df = df.drop(test_df.index).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return train_df, test_df

# ===================== 列名清洗函数 =====================
def clean_feature_name(name: str):
    name = re.sub(r'[()℃μmΩ²h eV+-]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')

# ===================== ±25%准确率计算 =====================
def calc_asr_acc_25(y_true, y_pred):
    relative_error = np.abs(y_pred - y_true) / (np.abs(y_true) + 1e-8)
    acc = (relative_error <= 0.25).mean() * 100
    return acc

In [7]:
# ===================== 空值版特征工程（LightGBM专用） =====================
class NullPreprocessor:
    def __init__(self, rare_threshold: float = 0.02, add_missing_flag: bool = True):
        self.rare_threshold = rare_threshold
        self.add_missing_flag = add_missing_flag
        
        self.rare_category_map_ = {}
        self.categorical_dummy_cols_ = []
        self.feature_names_ = []
        
    def fit(self, train_df: pd.DataFrame):
        df = train_df.copy()
        df = df.drop(columns=DROP_COLS, errors="ignore")
        
        for col in CATEGORICAL_COLS:
            col_series = df[col].fillna("缺失").astype(str).str.strip()
            value_counts = col_series.value_counts(normalize=True)
            rare_cats = value_counts[value_counts < self.rare_threshold].index.tolist()
            cat_map = {cat: "其他" if cat in rare_cats else cat for cat in value_counts.index}
            # 修复：补全字典赋值语句
            self.rare_category_map_[col] = cat_map
        
        temp_df = df.copy()
        for col in CATEGORICAL_COLS:
            temp_df[col] = temp_df[col].fillna("缺失").astype(str).str.strip()
            temp_df[col] = temp_df[col].map(self.rare_category_map_[col]).fillna("其他")
        
        temp_encoded = pd.get_dummies(temp_df[CATEGORICAL_COLS], prefix=CATEGORICAL_COLS, dtype=int)
        self.categorical_dummy_cols_ = temp_encoded.columns.tolist()
        return self
    
    def transform(self, df: pd.DataFrame):
        df = df.copy()
        df = df.drop(columns=DROP_COLS, errors="ignore")
        
        for col in CATEGORICAL_COLS:
            df[col] = df[col].fillna("缺失").astype(str).str.strip()
            df[col] = df[col].map(self.rare_category_map_[col]).fillna("其他")
        
        if self.add_missing_flag:
            for col in NUMERIC_COLS:
                df[f"{col}_is_missing"] = df[col].isna().astype(int)
        
        # 物理衍生特征
        df["asr_temp_K"] = df[COL_ASR_TEMP] + 273.15
        df["inv_asr_temp_K"] = 1 / (df["asr_temp_K"] + 1e-8)
        df["delta_phase_air_asr"] = df[COL_ASR_TEMP] - df[COL_PHASE_TRANS_AIR]
        df["thickness_asr_temp"] = df[COL_THICKNESS] * df[COL_ASR_TEMP]
        df["asr_time_init"] = df[COL_ASR_TIME] * df[COL_ASR_INIT]
        df["Ea_over_asr_temp"] = df[COL_ACTIVATION_E] / (df["asr_temp_K"] + 1e-8)
        df = df.replace([np.inf, -np.inf], np.nan)
        
        dummies = pd.get_dummies(df[CATEGORICAL_COLS], prefix=CATEGORICAL_COLS, dtype=int)
        for col in self.categorical_dummy_cols_:
            if col not in dummies.columns:
                dummies[col] = 0
        dummies = dummies[self.categorical_dummy_cols_]
        
        df_final = pd.concat([df.drop(columns=CATEGORICAL_COLS), dummies], axis=1)
        feature_cols = [col for col in df_final.columns if col not in [LABEL_COL, COL_COATING]]
        X_df = df_final[feature_cols].copy()

        X_df.columns = [clean_feature_name(c) for c in X_df.columns]
        self.feature_names_ = X_df.columns.tolist()

        y_df = df_final[LABEL_COL].copy()
        y_df = np.log1p(y_df.clip(lower=1e-6))
        return X_df, y_df
    
    def fit_transform_train(self, train_df: pd.DataFrame):
        self.fit(train_df)
        return self.transform(train_df)
    
    @staticmethod
    def inverse_transform_label(y_pred_log):
        return np.expm1(y_pred_log)


# ===================== 均值填充版特征工程（神经网络专用） =====================
class MeanFillPreprocessor:
    def __init__(self, rare_threshold: float = 0.02, add_missing_flag: bool = True):
        self.rare_threshold = rare_threshold
        self.add_missing_flag = add_missing_flag
        
        self.group_mean_dict_ = {}
        self.global_mean_ = None
        self.rare_category_map_ = {}
        self.categorical_dummy_cols_ = []
        self.feature_names_ = []
        
    def fit(self, train_df: pd.DataFrame):
        df = train_df.copy()
        df = df.drop(columns=DROP_COLS, errors="ignore")
        
        for formula, group in df.groupby(COL_COATING):
            self.group_mean_dict_[formula] = group[NUMERIC_COLS].mean()
        self.global_mean_ = df[NUMERIC_COLS].mean()
        
        for col in CATEGORICAL_COLS:
            col_series = df[col].fillna("缺失").astype(str).str.strip()
            value_counts = col_series.value_counts(normalize=True)
            rare_cats = value_counts[value_counts < self.rare_threshold].index.tolist()
            cat_map = {cat: "其他" if cat in rare_cats else cat for cat in value_counts.index}
            self.rare_category_map_[col] = cat_map
        
        temp_df = df.copy()
        for col in CATEGORICAL_COLS:
            temp_df[col] = temp_df[col].fillna("缺失").astype(str).str.strip()
            temp_df[col] = temp_df[col].map(self.rare_category_map_[col]).fillna("其他")
        
        temp_encoded = pd.get_dummies(temp_df[CATEGORICAL_COLS], prefix=CATEGORICAL_COLS, dtype=int)
        self.categorical_dummy_cols_ = temp_encoded.columns.tolist()
        return self
    
    def _fill_numeric(self, df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
        df = df.copy()
        if is_train:
            for formula, group in df.groupby(COL_COATING):
                mask = df[COL_COATING] == formula
                if formula in self.group_mean_dict_:
                    df.loc[mask, NUMERIC_COLS] = df.loc[mask, NUMERIC_COLS].fillna(self.group_mean_dict_[formula])
        df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(self.global_mean_)
        return df
    
    def transform(self, df: pd.DataFrame, is_train: bool = False):
        df = df.copy()
        df = df.drop(columns=DROP_COLS, errors="ignore")
        
        for col in CATEGORICAL_COLS:
            df[col] = df[col].fillna("缺失").astype(str).str.strip()
            df[col] = df[col].map(self.rare_category_map_[col]).fillna("其他")
        
        if self.add_missing_flag:
            for col in NUMERIC_COLS:
                df[f"{col}_is_missing"] = df[col].isna().astype(int)
        
        df = self._fill_numeric(df, is_train)
        # 物理衍生特征
        df["asr_temp_K"] = df[COL_ASR_TEMP] + 273.15
        df["inv_asr_temp_K"] = 1 / (df["asr_temp_K"] + 1e-8)
        df["delta_phase_air_asr"] = df[COL_ASR_TEMP] - df[COL_PHASE_TRANS_AIR]
        df["thickness_asr_temp"] = df[COL_THICKNESS] * df[COL_ASR_TEMP]
        df["asr_time_init"] = df[COL_ASR_TIME] * df[COL_ASR_INIT]
        df["Ea_over_asr_temp"] = df[COL_ACTIVATION_E] / (df["asr_temp_K"] + 1e-8)
        df = df.replace([np.inf, -np.inf], np.nan)
        
        dummies = pd.get_dummies(df[CATEGORICAL_COLS], prefix=CATEGORICAL_COLS, dtype=int)
        for col in self.categorical_dummy_cols_:
            if col not in dummies.columns:
                dummies[col] = 0
        dummies = dummies[self.categorical_dummy_cols_]
        
        df_final = pd.concat([df.drop(columns=CATEGORICAL_COLS), dummies], axis=1)
        feature_cols = [col for col in df_final.columns if col not in [LABEL_COL, COL_COATING]]
        X_df = df_final[feature_cols].copy()

        X_df.columns = [clean_feature_name(c) for c in X_df.columns]
        self.feature_names_ = X_df.columns.tolist()

        y_df = df_final[LABEL_COL].copy()
        y_df = np.log1p(y_df.clip(lower=1e-6))
        return X_df, y_df
    
    def fit_transform_train(self, train_df: pd.DataFrame):
        self.fit(train_df)
        return self.transform(train_df, is_train=True)
    
    @staticmethod
    def inverse_transform_label(y_pred_log):
        return np.expm1(y_pred_log)

In [8]:
file_path = r'C:\Users\Lenovo\Desktop\田老师项目\ASR数据库最新.xlsx'
# 修复：正确参数为sheet_name
df_raw = pd.read_excel(file_path, sheet_name='简易ASR模型')

# 清理末尾空列
df_raw = df_raw.drop(columns=["Unnamed: 15", "Unnamed: 16"], errors="ignore")

# 过滤标签空值
df_raw = df_raw.dropna(subset=[COL_COATING, COL_ASR], how="any").reset_index(drop=True)

# 数据校验打印
print("="*80)
print("✅ 已读取工作表：简易ASR模型")
print("="*80)
print(f"清洗后有效样本数：{len(df_raw)}")
print(f"唯一涂层配方数：{df_raw[COL_COATING].nunique()}")
print("="*80)

✅ 已读取工作表：简易ASR模型
清洗后有效样本数：372
唯一涂层配方数：148


In [9]:
train_raw, test_raw = split_by_formula(df_raw, test_size=0.2, random_state=42)

print(f"训练集配方数：{train_raw[COL_COATING].nunique()}，样本数：{len(train_raw)}")
print(f"测试集配方数：{test_raw[COL_COATING].nunique()}，样本数：{len(test_raw)}")
print(f"配方重叠数：{len(set(train_raw[COL_COATING]) & set(test_raw[COL_COATING]))} （允许同一涂层分布在训练/测试集）")
print("="*80)

训练集配方数：128，样本数：298
测试集配方数：50，样本数：74
配方重叠数：30 （允许同一涂层分布在训练/测试集）


In [10]:
# 6.1 空值版：LightGBM专用
null_preprocessor = NullPreprocessor(rare_threshold=0.02, add_missing_flag=True)
X_train_null, y_train_log = null_preprocessor.fit_transform_train(train_raw)
X_test_null, y_test_log = null_preprocessor.transform(test_raw)

# 6.2 均值填充版：神经网络专用
mean_preprocessor = MeanFillPreprocessor(rare_threshold=0.02, add_missing_flag=True)
X_train_fill, y_train_log_fill = mean_preprocessor.fit_transform_train(train_raw)
X_test_fill, y_test_log_fill = mean_preprocessor.transform(test_raw, is_train=False)

# 6.3 神经网络特征标准化
scaler = StandardScaler()
X_train_fill_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_fill),
    columns=X_train_fill.columns
)
X_test_fill_scaled = pd.DataFrame(
    scaler.transform(X_test_fill),
    columns=X_test_fill.columns
)

# 测试集真实标签（原始单位）
y_test_real = null_preprocessor.inverse_transform_label(y_test_log.values)

print("特征工程处理完成")
print(f"树模型特征维度：{X_train_null.shape[1]}")
print(f"神经网络特征维度：{X_train_fill_scaled.shape[1]}")
print("="*80)

特征工程处理完成
树模型特征维度：37
神经网络特征维度：37


In [11]:
import importlib
import sys

sys.path.append(r"D:\尖晶石涂层ASR电导率预测")
import 基线ASR单预测 as train_module
importlib.reload(train_module)
from 基线ASR单预测 import train_lightgbm_asr, train_sklearn_mlp_asr, train_pytorch_asr

print("训练函数导入成功")

训练函数导入成功


In [12]:
res_lgb = train_lightgbm_asr(X_train_null, y_train_log, X_test_null, y_test_log)
y_pred_lgb_log = res_lgb["model"].predict(X_test_null)
y_pred_lgb_real = null_preprocessor.inverse_transform_label(y_pred_lgb_log)

acc_lgb = calc_asr_acc_25(y_test_real, y_pred_lgb_real)
r2_lgb = r2_score(y_test_real, y_pred_lgb_real)
mae_lgb = mean_absolute_error(y_test_real, y_pred_lgb_real)

print("="*60)
print("【强基线·空值特征】LightGBM")
print(f"R²: {r2_lgb:.4f} | MAE: {mae_lgb:.4f}")
print(f"ASR Acc@±25%: {acc_lgb:.2f}%")
print("="*60)

【强基线·空值特征】LightGBM
R²: 0.6179 | MAE: 4.0026
ASR Acc@±25%: 60.81%


In [13]:
res_mlp = train_sklearn_mlp_asr(X_train_fill_scaled, y_train_log_fill, X_test_fill_scaled, y_test_log_fill)
y_pred_mlp_log = res_mlp["model"].predict(X_test_fill_scaled)
y_pred_mlp_real = mean_preprocessor.inverse_transform_label(y_pred_mlp_log)

acc_mlp = calc_asr_acc_25(y_test_real, y_pred_mlp_real)
r2_mlp = r2_score(y_test_real, y_pred_mlp_real)
mae_mlp = mean_absolute_error(y_test_real, y_pred_mlp_real)

print("="*60)
print("【传统MLP·填充特征】Sklearn_MLP")
print(f"R²: {r2_mlp:.4f} | MAE: {mae_mlp:.4f}")
print(f"ASR Acc@±25%: {acc_mlp:.2f}%")
print("="*60)

【传统MLP·填充特征】Sklearn_MLP
R²: 0.2387 | MAE: 5.3891
ASR Acc@±25%: 47.30%


C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but MLPRegressor was fitted without feature names
  warnings.warn(


In [14]:
res_pytorch = train_pytorch_asr(
    X_train_fill_scaled, y_train_log_fill, 
    X_test_fill_scaled, y_test_log_fill,
    scaler=scaler,
    thickness_col_name="涂层厚度(μm)X"
)

X_te_tensor = torch.tensor(np.asarray(X_test_fill_scaled), dtype=torch.float32)
res_pytorch["model"].eval()
with torch.no_grad():
    y_pred_norm_pytorch = res_pytorch["model"](X_te_tensor).numpy()

y_pred_pytorch_log = y_pred_norm_pytorch * res_pytorch["y_std"] + res_pytorch["y_mean"]
y_pred_pytorch_real = mean_preprocessor.inverse_transform_label(y_pred_pytorch_log)

acc_pyt = calc_asr_acc_25(y_test_real, y_pred_pytorch_real)
r2_pyt = r2_score(y_test_real, y_pred_pytorch_real)
mae_pyt = mean_absolute_error(y_test_real, y_pred_pytorch_real)

print("="*60)
print("【PyTorch·填充特征】PyTorch_ASR")
print(f"R²: {r2_pyt:.4f} | MAE: {mae_pyt:.4f}")
print(f"ASR Acc@±25%: {acc_pyt:.2f}%")
print("="*60)

【PyTorch·填充特征】PyTorch_ASR
R²: 0.2404 | MAE: 4.3091
ASR Acc@±25%: 22.97%


以下是参数优化后模型与基线模型对比：

In [15]:
import importlib
import sys

sys.path.append(r"D:\尖晶石涂层ASR电导率预测")
import ASR单预测调优 as opt_module
importlib.reload(opt_module)
from ASR单预测调优 import (
    bayes_opt_lightgbm_asr
)

print("✅ 贝叶斯超参优化训练函数导入成功")

✅ 贝叶斯超参优化训练函数导入成功


In [16]:
# 运行贝叶斯调参
res_lgb_opt = bayes_opt_lightgbm_asr(
    X_train_null, y_train_log,
    X_test_null, y_test_log,
    n_trials=60
)

# 反变换回原始数值空间计算指标
y_pred_lgb_opt_log = res_lgb_opt["model"].predict(X_test_null)
y_pred_lgb_opt_real = null_preprocessor.inverse_transform_label(y_pred_lgb_opt_log)

acc_lgb_opt = calc_asr_acc_25(y_test_real, y_pred_lgb_opt_real)
r2_lgb_opt = r2_score(y_test_real, y_pred_lgb_opt_real)
mae_lgb_opt = mean_absolute_error(y_test_real, y_pred_lgb_opt_real)

print("="*70)
print("【强基线·空值特征】LightGBM 贝叶斯调参结果")
print("="*70)
print("📌 最优超参数：")
for k, v in res_lgb_opt["best_params"].items():
    print(f"   {k}: {v}")
print("-"*70)
print(f"原始空间 R²: {r2_lgb_opt:.4f} | MAE: {mae_lgb_opt:.4f}")
print(f"ASR Acc@±25%: {acc_lgb_opt:.2f}%")
print("="*70)

【强基线·空值特征】LightGBM 贝叶斯调参结果
📌 最优超参数：
   n_estimators: 417
   max_depth: 8
   learning_rate: 0.024650630247887214
   num_leaves: 55
   subsample: 0.8669527615096319
   colsample_bytree: 0.8260389731328149
   reg_alpha: 1.3339745742348045e-05
   reg_lambda: 0.0001115434184352848
----------------------------------------------------------------------
原始空间 R²: 0.6672 | MAE: 3.8702
ASR Acc@±25%: 59.46%


In [17]:
# ===================== 新Cell: SHAP分析 + 约束筛选 + Top 20涂层推荐 =====================
# 前提：前面所有cell已运行，df_raw、null_preprocessor、res_lgb_opt、X_train_null 等变量已存在
import shap

# ---------- 1. SHAP 特征重要性（无图，直接打印） ----------
print("="*60)
print("【SHAP 特征重要性分析】")
print("="*60)

explainer = shap.TreeExplainer(res_lgb_opt["model"])
shap_values = explainer.shap_values(X_train_null)

fi = pd.DataFrame({
    "feature": X_train_null.columns,
    "importance": np.abs(shap_values).mean(axis=0)
}).sort_values("importance", ascending=False)

print("Top 10 特征:")
for _, row in fi.head(10).iterrows():
    print(f"  {row['importance']:.6f} | {row['feature']}")

# ---------- 2. 预测全部样本 ----------
X_all, _ = null_preprocessor.transform(df_raw)
y_pred = null_preprocessor.inverse_transform_label(
    res_lgb_opt["model"].predict(X_all)
)

# ---------- 3. 构建评估表（使用已有列，直接约束筛选） ----------
eval_df = pd.DataFrame({
    COL_COATING: df_raw[COL_COATING].values,
    COL_THICKNESS: df_raw[COL_THICKNESS].values,
    COL_ASR_TEMP: df_raw[COL_ASR_TEMP].values,
    COL_ASR_TIME: df_raw[COL_ASR_TIME].values,
    COL_ACTIVATION_E: df_raw[COL_ACTIVATION_E].values,
    COL_SYNTHESIS: df_raw[COL_SYNTHESIS].values,
    COL_PREPARATION: df_raw[COL_PREPARATION].values,
    COL_SUBSTRATE: df_raw[COL_SUBSTRATE].values,
    COL_ASR: df_raw[COL_ASR].values,
    "ASR_pred": y_pred,
})

# 硬约束：预测腐蚀后 ASR ≤ 50
feasible = eval_df[eval_df["ASR_pred"] <= 50].copy()

# ---------- 4. 按涂层配方聚合，采集函数排名 ----------
# 采集函数：LCB（Lower Confidence Bound）= 预测均值 - 1×标准差
# 兼顾"预测值低"与"预测稳定性高"
grouped = feasible.groupby(COL_COATING).apply(
    lambda g: pd.Series({
        "pred_mean": g["ASR_pred"].mean(),
        "pred_std": g["ASR_pred"].std(),
        "pred_min": g["ASR_pred"].min(),
        "count": len(g),
        "true_min": g[COL_ASR].min(),
        "thickness": g[COL_THICKNESS].iloc[0],
        "asr_temp": g[COL_ASR_TEMP].iloc[0],
        "asr_time": g[COL_ASR_TIME].iloc[0],
        "Ea": g[COL_ACTIVATION_E].iloc[0],
        "synthesis": g[COL_SYNTHESIS].iloc[0],
        "preparation": g[COL_PREPARATION].iloc[0],
        "substrate": g[COL_SUBSTRATE].iloc[0],
    })
).reset_index()

grouped["LCB"] = grouped["pred_mean"] - 1.0 * grouped["pred_std"]

# 约束加分（MCO标准 + 文献补充）
grouped["constraint_score"] = 0
grouped.loc[(grouped["thickness"] >= 5) & (grouped["thickness"] <= 20), "constraint_score"] += 1
grouped.loc[grouped["asr_time"] >= 1000, "constraint_score"] += 1
grouped.loc[abs(grouped["asr_temp"] - 650) <= 50, "constraint_score"] += 1
grouped.loc[grouped["Ea"] <= 0.5, "constraint_score"] += 1

# 最终排名：LCB越低越好，约束得分越高越好
grouped["rank_score"] = grouped["LCB"] - grouped["constraint_score"] * 3

# ---------- 5. Top 20 输出 ----------
top20 = grouped.sort_values("rank_score").head(20)

print("\n")
print("="*60)
print("【Top 20 高性能MCO涂层推荐】")
print("="*60)
print(f"{'Rank':>4} | {'涂层':<20} | {'LCB':>7} | {'预测ASR':>8} | {'真实ASR':>8} | {'约束':>4}")
print("-"*60)

for i, (_, row) in enumerate(top20.iterrows()):
    print(f"{i+1:>4} | {row[COL_COATING]:<20} | {row['LCB']:>7.2f} | {row['pred_min']:>8.2f} | {row['true_min']:>8.2f} | {row['constraint_score']:>4.0f}")

print("-"*60)
print("采集函数: LCB = pred_mean - 1×pred_std（Lower Confidence Bound）")
print("约束加分: 厚度5-20μm、时间≥1000h、温度≈650℃、活化能≤0.5eV")
print("="*60)

# ---------- 6. Top 20 涂层详细工艺参数 ----------
print("\n")
print("="*60)
print("【Top 20 涂层详细工艺参数】")
print("="*60)

for i, (_, row) in enumerate(top20.iterrows()):
    print(f"\n--- Rank {i+1}: {row[COL_COATING]} ---")
    print(f"  合成方法: {row['synthesis']}")
    print(f"  制备方法: {row['preparation']}")
    print(f"  基体:     {row['substrate']}")
    print(f"  厚度:     {row['thickness']:.1f} μm")
    print(f"  测试温度: {row['asr_temp']:.1f} ℃")
    print(f"  测试时间: {row['asr_time']:.1f} h")
    print(f"  活化能:   {row['Ea']:.4f} eV")
    print(f"  预测ASR:  {row['pred_min']:.2f} mΩ·cm²")
    print(f"  真实ASR:  {row['true_min']:.2f} mΩ·cm²")
    print(f"  约束得分: {row['constraint_score']:.0f}/4")

【SHAP 特征重要性分析】
Top 10 特征:
  0.446467 | 起始ASR
  0.104398 | 涂层厚度_X
  0.086115 | ASR测试时间
  0.074729 | 合成方法_缺失
  0.054289 | 相转温度（空气）（_）
  0.053967 | asr_ti_init
  0.046111 | ASR测试温度_X
  0.039852 | 连接体基体X_Crof_r_22_APU
  0.029560 | 电导率活化能
  0.025516 | t_ickn_ss_asr_t_p


【Top 20 高性能MCO涂层推荐】
Rank | 涂层                   |     LCB |    预测ASR |    真实ASR |   约束
------------------------------------------------------------
   1 | MnCo1.5Cu0.5O4       |    0.51 |     0.87 |     0.17 |    2
   2 | Mn1.3Co1.3Cu0.3Y0.1O4 |    0.76 |     0.96 |     0.08 |    2
   3 | CuMn1.9Fe0.1O4       |    2.54 |     3.49 |     4.50 |    2
   4 | Mn2CuO4              |    2.81 |     3.24 |     1.70 |    2
   5 | Mn1.9CuFe0.1O4       |    3.19 |     3.47 |     2.70 |    2
   6 | Mn1.7CuFe0.3O4       |    3.20 |     3.54 |     2.14 |    2
   7 | Ni0.17Mn0.73Co2.1O4  |    0.70 |     2.07 |     1.10 |    1
   8 | Mn1.8CuO4            |    0.71 |     4.26 |     4.60 |    1
   9 | Mn1.2Co1.2Cu0.5Y0.1O4 |    0.92 |     0.9

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21736\2067578835.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = feasible.groupby(COL_COATING).apply(
